In [ ]:
SODA_URL = "https://data.lacity.org/resource/d5tf-ez2w.json"
DATE_FROM = "2022-01-01"
BATCH_SIZE = 50000
FIELDS = ["dr_no","date_occ","time_occ","area_name","crm_cd_desc",
          "vict_age","vict_sex","vict_descent","premis_desc","location",
          "cross_street","lat","lon","mocodes"]
print("Config OK")

In [ ]:
import requests
all_records = []
offset = 0
while True:
    params = {
        "": "date_occ>" + chr(39) + DATE_FROM + "T00:00:00" + chr(39),
        "": str(BATCH_SIZE),
        "": str(offset),
        "": ",".join(FIELDS),
        "": "date_occ DESC",
    }
    r = requests.get(SODA_URL, params=params, timeout=180)
    r.raise_for_status()
    batch = r.json()
    if not batch: break
    all_records.extend(batch)
    print(len(all_records), "records...")
    if len(batch) < BATCH_SIZE: break
    offset += BATCH_SIZE
print("Fetched:", len(all_records))

In [ ]:
rows = [tuple(str(rec.get(f, "") or "") for f in FIELDS) for rec in all_records]
print("Normalized:", len(rows))

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql import functions as F
schema = StructType([StructField(f, StringType(), True) for f in FIELDS])
df = spark.createDataFrame(rows, schema)
print("DataFrame:", df.count(), "rows")

In [ ]:
df2 = df.withColumn("crash_year", F.year(F.to_date("date_occ")))
df2.write.format("delta").mode("overwrite").partitionBy("crash_year").saveAsTable("bronze_crashes")
print("Wrote bronze_crashes")
spark.sql("SELECT crash_year, count(*) cnt FROM bronze_crashes GROUP BY 1 ORDER BY 1").show()

In [ ]:
print("BRONZE COMPLETE:", spark.table("bronze_crashes").count(), "rows")